# 🤖 AI-Powered Study Assistant
### RAG + Multi-Agent System | LangChain · CrewAI · Groq · ChromaDB

## About
This project demonstrates a **RAG pipeline (Retrieval-Augmented Generation)** 
combined with a **Multi-Agent System (Agent-to-Agent Communication)**.

Real university lecture slides (344 pages, 7 PDFs) serve as the knowledge base.
Two specialized AI agents collaborate:
- **Agent 1 (Classifier)** – classifies the query by topic and complexity
- **Agent 2 (Solver)** – searches the vector database and formulates an answer

## Tech Stack
| Component | Technology |
|---|---|
| LLM | Llama 4 via Groq API |
| RAG Framework | LangChain |
| Agent Framework | CrewAI |
| Vector Database | ChromaDB |
| Embeddings | HuggingFace (all-MiniLM-L6-v2) |
| Environment | Kaggle Notebook |


# ── Cell 1: Install Dependencies ────────────────────────────────────────


In [1]:
import subprocess, sys

# Install all required packages with pinned versions for compatibility
# stderr=DEVNULL suppresses pip dependency warnings (cosmetic only)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "langchain==0.3.25",
    "langchain-core==0.3.83",        # pinned: newer versions break compatibility
    "langchain-community==0.3.24",
    "langchain-groq==0.3.2",
    "langchain-huggingface==0.1.2",
    "chromadb==0.6.3",
    "crewai==0.114.0",
    "pypdf==5.4.0",
    "sentence-transformers",
    "--upgrade"
], stderr=subprocess.DEVNULL)

# Verify installed versions
import langchain, langchain_core, langchain_groq
print(f"✅ langchain: {langchain.__version__}")
print(f"✅ langchain-core: {langchain_core.__version__}")
print(f"✅ langchain-groq: {langchain_groq.__version__}")
print("🎉 All packages ready!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.5/285.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# ── Cell 2: API Setup & Model Configuration ───────────────────────


In [11]:
# Load the Groq API key securely from Kaggle Secrets and configure the LLM and embeddings

import os
import warnings
import logging

from kaggle_secrets import UserSecretsClient
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# Read the Kaggle secret 'AI-assistant_KEY' and expose its value as GROQ_API_KEY
os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("AI-assistant_KEY")

# LLM used for RAG answers and agent reasoning
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.0,
)

# Local embedding model for semantic similarity search
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

warnings.resetwarnings()

# Quick connectivity test
response = llm.invoke("Say hello in one word.")

print(f"✅ Groq API connected. Response: {response.content}")
print("✅ Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Groq API connected. Response: Hello.
✅ Embedding model loaded.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# ── Cell 3: Locate PDF Files ─────────────────────────────────────────


In [12]:
import glob, os

# Path to uploaded PDF dataset in Kaggle
# To use your own PDFs: upload via "Add Data → Upload" and adjust this path
base_path = "/kaggle/input/datasets/divinefirstyolo/folien-fh/"
all_pdfs = glob.glob(base_path + "*.pdf")

print(f"📂 Found PDFs ({len(all_pdfs)} files):")
for pdf in all_pdfs:
    print(f"  → {pdf.split('/')[-1]}")

📂 Found PDFs (7 files):
  → 2025_10_FolienLehrveranstaltung.pdf
  → AI25 Part1 AI Motivation and Definition.pdf
  → AI25 Part2 Cloud AI Computer Vision.pdf
  → AI25 Part3 LLM genKI.pdf
  → LV_IT-Security_1_2024.pdf
  → LV_IT-Security_2_2024.pdf
  → Methoden des Technologiemanagements.pdf


# ── Cell 4: Build RAG Pipeline & Vector Database ────────────


In [13]:
import warnings, logging
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Suppress ChromaDB telemetry warnings (known bug in chromadb 0.6.3)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb").setLevel(logging.ERROR)

# Step 1: Load all PDFs page by page
all_docs = []
for path in all_pdfs:
    try:
        loader = PyPDFLoader(path)
        all_docs.extend(loader.load())
        print(f"✅ {path.split('/')[-1]}")
    except Exception as e:
        print(f"⚠️ Skipped: {path.split('/')[-1]} → {e}")

print(f"\n📚 {len(all_docs)} pages loaded")

# Step 2: Split documents into chunks for semantic search
# chunk_size=800: balanced between context and precision
# chunk_overlap=100: prevents context loss at chunk boundaries
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(all_docs)
print(f"✂️  {len(chunks)} chunks created")

# Step 3: Embed chunks and store in ChromaDB vector database
# Each chunk is converted to a vector → enables semantic similarity search
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✅ Vector database ready!")

✅ 2025_10_FolienLehrveranstaltung.pdf
✅ AI25 Part1 AI Motivation and Definition.pdf
✅ AI25 Part2 Cloud AI Computer Vision.pdf
✅ AI25 Part3 LLM genKI.pdf
✅ LV_IT-Security_1_2024.pdf
✅ LV_IT-Security_2_2024.pdf
✅ Methoden des Technologiemanagements.pdf

📚 344 pages loaded
✂️  412 chunks created


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/chromadb/types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector database ready!


# ── Cell 5: RAG Test (Single Query) ──────────────────────────────────────────


In [14]:
from langchain.chains import RetrievalQA

# Build QA chain: retriever fetches relevant chunks → LLM formulates answer
# chain_type="stuff": all retrieved chunks are passed together to the LLM
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

# Test query – change this to any topic covered in your PDFs
query = "What is a Large Language Model?"

result = qa_chain.invoke({"query": query})

print("🧠 ANSWER:")
print(result["result"])
print("\n📎 SOURCES:")
for doc in result["source_documents"]:
    print(f"  → Page {doc.metadata.get('page', '?')} | {doc.metadata.get('source', '?').split('/')[-1]}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/chromadb/types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datet

🧠 ANSWER:
A **Large Language Model (LLM)** is a type of foundation model—a powerful machine‑learning system that has been pre‑trained on massive amounts of text data. During pre‑training, the model learns to predict the next word (or token) in a sentence, which lets it capture patterns, syntax, and even some world knowledge from the data it was exposed to.

Key points:

| Feature | What it means |
|---------|---------------|
| **Scale** | LLMs contain billions of parameters (the “weights” the model uses). The larger the parameter count, the more nuanced the model’s internal representations can be. |
| **Pre‑training** | The model is trained on a broad, diverse corpus (books, websites, forums, Wikipedia, etc.) using a causal language‑model objective (predict the next token). |
| **Foundation model** | Once pre‑trained, the same model can be fine‑tuned or prompted for many downstream tasks—text generation, summarization, translation, question answering, code generation, etc.—without need

# ── Cell 6: Multi-Agent Chat UI (CrewAI + ipywidgets) ──────────────────


In [24]:
# AI-Powered Study Assistant: classifier agent and RAG knowledge assistant

from ipywidgets import widgets
from IPython.display import display, clear_output
from crewai import Agent, Task, Crew
from langchain_groq import ChatGroq

# Model used by CrewAI agents
CREWAI_MODEL = "groq/openai/gpt-oss-20b"

# Translation model: translates German questions into English for semantic search
translation_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.0,
)

# Agent 1: classifies the user question
classifier = Agent(
    role="Question Classifier",
    goal=(
        "Classify the user's question as either a technical question or a "
        "question about how the assistant works."
    ),
    backstory=(
        "You are an experienced AI assistant. You understand user questions "
        "and identify whether they ask about technical lecture content or "
        "about the assistant itself."
    ),
    llm=CREWAI_MODEL,
    verbose=False,
)

# Agent 2: answers technical questions using retrieved lecture material
knowledge_assistant = Agent(
    role="Knowledge Assistant",
    goal=(
        "Answer technical questions using the retrieved lecture material and "
        "provide source references."
    ),
    backstory=(
        "You are a knowledgeable assistant for university AI lectures. "
        "You answer only from the retrieved PDF context and explain where the "
        "information comes from."
    ),
    llm=CREWAI_MODEL,
    verbose=False,
)

# UI elements
title = widgets.HTML(
    value="<h2>AI-Powered Study Assistant</h2>"
)

query_input = widgets.Textarea(
    placeholder="Ask a question in German or English...",
    layout=widgets.Layout(width="100%", height="90px"),
)

ask_button = widgets.Button(
    description="Start Agents",
    button_style="primary",
)

output = widgets.Output()


def on_button_click(button):
    query = query_input.value.strip()

    with output:
        clear_output()

        if not query:
            print("Please enter a question.")
            return

        # Agent 1: classify the question
        print("🤖 Agent 1: Question Classifier")
        print("⏳ Classifying question...")

        classification_task = Task(
            description=(
                "Classify the following question as either 'technical' or "
                "'assistant_information'. Return only one word.\n\n"
                f"Question: {query}"
            ),
            expected_output="One word: technical or assistant",
            agent=classifier,
        )

        classification_result = Crew(
            agents=[classifier],
            tasks=[classification_task],
            verbose=False,
        ).kickoff()

        classification = str(classification_result).strip().lower()

        print(f"Classification: {classification}")
        print()

        # Translate German questions into English for semantic search
        print("🌐 Translating question for semantic search...")
        translation = translation_llm.invoke(
            (
                "Translate this question to English. Return only the translated "
                f"question, nothing else: {query}"
            )
        ).content

        english_query = str(translation).strip()
        print(f"English query: {english_query}")
        print()

        # Agent 2: retrieve relevant PDF passages
        print("📚 Agent 2: Knowledge Assistant")
        print("⏳ Searching PDFs...")

        relevant_chunks = retriever.invoke(english_query)
        context = "\n\n".join(chunk.page_content for chunk in relevant_chunks)

        print(f"📚 Found {len(relevant_chunks)} relevant passages.")
        print()

        # Generate the final answer
        answer = llm.invoke(
            (
                "Answer the question using only the provided lecture context. "
                "If the question asks whether you use the internet, explain that you "
                "do not have internet access and answer only from the provided PDF "
                "context. "
                "If the context does not contain enough information for another "
                "question, say that the answer is not covered by the available "
                "material. "
                "Always answer in the same language as the user's original "
                "question.\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {query}"
            )
        ).content

        print("🧠 ANSWER:")
        print(answer)
        print()

        print("📎 SOURCES:")

        for chunk in relevant_chunks:
            page = chunk.metadata.get("page", "?")
            source = chunk.metadata.get("source", "?").split("/")[-1]
            print(f"  → Page {page} | {source}")


ask_button.on_click(on_button_click)

display(title, query_input, ask_button, output)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


HTML(value='<h2>AI-Powered Study Assistant</h2>')

Textarea(value='', layout=Layout(height='90px', width='100%'), placeholder='Ask a question in German or Englis…

Button(button_style='primary', description='Start Agents', style=ButtonStyle())

Output()